# Wedding Planner

This notebook applies advanced concepts from context and state, mcp and multi-agent systems to build a destination wedding planning assistant. The assistant will be a multi-agent system with a coordinating agent and three sub-agents: a travel agent, a venue agent and a DJ (playlist) agent.

- Travel agent: will utilize an external MCP server to find flights to and from a selected destination
- Venue agent: will use the context of the selected destination and Tavily API to search for venues
- DJ: will use context about music preferences and the destination to develop an appropriate playlist for the occasion

#### Design

Orchestrator:
- The orchestrator will be configured with state and memory to capture preferences and decisions about the destination, flight, venue and music. The agent's state will have the following variables:
  - Destination
  - Flight preferences
  - Flight
  - Venue preferences
  - Venue
  - Wedding size
  - Music preferences
- The orchestrator agent will pass these state variables to the sub-agents as context when calling them
- The orchestator will have the following tools
  - Call travel agent
  - Call dj agent
  - Update destination preferences
  - Update venue preferences
  - Update music preferences
  - Update flight preferences
  - Update flight
  - Update venue

Travel Agent:
- The travel agent will have the following tools:
  - Search for flights
  - Evaluate flights
- Travel agent will inherit context from the orchestator agent when calling tools

DJ Agent:
- The DJ agent will have the following tools:
  - Search for music
  - Create playlist
- DJ agent will inherit context from the orchestrator agent when calling tools

## Building the Agentic System

In [1]:
# import modules
from langchain.agents import create_agent, AgentState
from langchain.tools import tool, ToolRuntime
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.messages import HumanMessage

from dotenv import load_dotenv

from dataclasses import dataclass

Creating an orchestrator state class to track the relevant variables

In [2]:
@dataclass
class weddingContext:
    destination: str
    wedding_date: str | None
    wedding_size: float | None
    departure_city: str | None



#flight_pref: str | None
#flight: str | None
#venue_pref: str | None
#venue_name: str | None
#music_pref: str | None

### Writing subagent tools

#### Using the Departi MCP Server for flights

In [3]:
kiwi_client = MultiServerMCPClient(
    {
        "travel_server": {
                "transport": "streamable_http",
                "url": "https://mcp.kiwi.com"
            }
    }
)

kiwi_tools = await kiwi_client.get_tools()

In [4]:
kiwi_agent = create_agent(
    model="claude-haiku-4-5",
    tools=kiwi_tools
)

Defining the subagent tools for performing actions and updating the orchestrator's state

In [ ]:
@tool
def read_destination(runtime: ToolRuntime) -> str:
    '''Reads the selected destination from the orchestrator's state'''
    try:
        return runtime.context.destination
    except:
        return "No destination could be found in state"

@tool
def read_wedding_size(runtime: ToolRuntime) -> str:
    '''Reads the planned size of the wedding from the orchestrator's state'''
    try:
        return runtime.context.wedding_size
    except:
        return "No wedding size could be found in state"

@tool
def read_wedding_date(runtime: ToolRuntime) -> str:
    '''Reads the planned date of the wedding from the orchestrator's state'''
    try:
        return runtime.context.wedding_date
    except:
        return "No wedding date could be found in state"

@tool
def read_depature_city(runtime: ToolRuntime) -> str:
    '''Reads the planned departure city of the wedding from the orchestrator's state'''
    try:
        return runtime.context.departure_city
    except:
        return "No departure city could be found in state"

In [12]:
# Creating the flight subagent
kiwi_agent = create_agent(
    model="claude-haiku-4-5",
    tools=kiwi_tools
)

#append(read_destination)

Create the main orchestrator agent to call each of the subagents

In [29]:
@tool
def call_kiwi_agent(destination: str, date: str, departure_city: str) -> str:
    """Tool to call the kiwi agent to find flights"""
    try:
        flights = kiwi_agent.ainvoke({"messages":HumanMessage(content=f"Get flights to {destination} from {departure_city} on {date}")})
        return flights.messages[-1],content
    except:
        "Failed to return flights using the Kiwi MCP server"

In [30]:
real_wedding_context = weddingContext(
    destination="Paris",
    wedding_date="2026-11-01",
    wedding_size=150,
    departure_city="Chicago"
)

orchestrator_prompt = """
You are a wedding planner that helps plan destination weddings.
Use the defined tools to read the context about the wedding and 
to call the Kiwi agent to find flights for the wedding
"""

wedding_planner = create_agent(
    model="claude-sonnet-5",
    tools=[call_kiwi_agent,
           read_wedding_size,
           read_destination,
           read_wedding_date,
           read_depature_city],
    checkpointer=InMemorySaver(),
    context_schema=weddingContext,
    system_prompt=orchestrator_prompt
)

In [31]:
test = wedding_planner.invoke(
    {'messages':HumanMessage(content="Help me find flights to my wedding")},
    {"configurable": {"thread_id": "1"}},
    context=real_wedding_context
)

/Users/samueljoseph/Documents/Programming/agent-lab/.venv/lib/python3.13/site-packages/pydantic/functional_validators.py:835: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=weddingContext(destinatio...eparture_city='Chicago'), input_type=weddingContext])
  function=lambda v, h: h(v), schema=original_schema
/Users/samueljoseph/Documents/Programming/agent-lab/.venv/lib/python3.13/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=weddingContext(destinatio...eparture_city='Chicago'), input_type=weddingContext])
  return self.__pydantic_serializer__.to_python(
/Users/samueljoseph/Documents/Programming/agent-lab/.venv/lib/python3.13/site-packages/langchain_core/tools/structured.py:97: RuntimeWarning: corouti

In [33]:
test

{'messages': [HumanMessage(content='Help me find flights to my wedding', additional_kwargs={}, response_metadata={}, id='f72d9428-eb04-470e-a34b-1d47c1710a80'),
  AIMessage(content=[{'signature': 'EukCCpABCBEYAipAGJhE+Aqw3/tmO1kj5uD+7BKNfYS62/5eZDDG7DMh33wXQZZV1CAfVdw134RYQlzIWimB+eQeHXHAmhnATin45zIPY2xhdWRlLXNvbm5ldC01OABCCHRoaW5raW5nWiQ3MzZmNjBiNS1hOGI0LTRlNjItOTZhNi03NDViMWVmNWY4NTaoAbzXsNQGEgy/yYXpTgUY0G3mwNwaDEZlhnTkILOT3YG7giIwLTLSiTAdPnIajMLgk6LLJ+g4UUGj3ZpAHTGf6JNvmKvbymLTyWrlX1Gpf3KQu5ZeKoUB3JSFuwH9yw0JEVhKJPWpAcnMXPUQk8KAa6CoPGq7P9BE67j94melH6KAdISSNCWJe+XPa4I67Uh4h/YynV7RDCMPsRAXghBwSxC3C+v9GYf7NyNLqynsDDakTgNHT7KpmD1rq1iFbABlEmV3QYHOD25XA7I0scjNMLdncRZeR3VOqHB7KxgB', 'thinking': '', 'type': 'thinking'}, {'id': 'toolu_01AsCuDKP42EzDwtezfgdyUH', 'caller': {'type': 'direct'}, 'input': {}, 'name': 'read_destination', 'type': 'tool_use'}, {'id': 'toolu_01A1xgMAvd9akYRrSBBbkSGM', 'caller': {'type': 'direct'}, 'input': {}, 'name': 'read_wedding_date', 'type': 'tool_use'}, {'id': '